# JSONB in PostgreSQL

## Overview
PostgreSQL's `JSONB` type stores JSON in a decomposed binary format, enabling powerful operators and GIN indexes that make JSON queries fast even on large tables.

**`JSON` vs `JSONB`:**

| Feature | `JSON` | `JSONB` |
|---|---|---|
| Storage | Raw text (preserves order/whitespace) | Binary (parsed, decomposed) |
| Write speed | Faster (no parsing) | Slower (parsing on insert) |
| Read speed | Slower (re-parses on every read) | Faster (binary access) |
| GIN indexing | Not supported | Supported |
| `@>`, `?` operators | Not supported | Supported |
| Duplicate keys | Keeps all (last wins on read) | Keeps last only |
| **Use when** | Storing verbatim external JSON | Anything you query or index |

**Rule: always use JSONB** unless you have a specific reason to preserve raw JSON text.

**These notebooks simulate PostgreSQL operators in Python** since no PostgreSQL server is available. All operator examples are valid PostgreSQL syntax.

---

In [ ]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

# Healthcare dataset: patients with JSON clinical data
conn.executescript("""
CREATE TABLE patients (
    patient_id   TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    demographics JSON NOT NULL,   -- age, dob, province, contact info
    clinical     JSON NOT NULL,   -- conditions, allergies, vitals history
    preferences  JSON             -- notification prefs, language, etc.
);

CREATE TABLE lab_results (
    lab_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    result_date TEXT NOT NULL,
    tests       JSON NOT NULL    -- array of {test, value, unit, ref_range, flag}
);

CREATE TABLE medications (
    med_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    prescribed  TEXT NOT NULL,
    details     JSON NOT NULL    -- drug, dose, frequency, prescriber, side_effects
);
""")

patients = [
  ('P001','Alice Ngata',
   json.dumps({'age':45,'dob':'1980-03-15','province':'NB','contact':{'phone':'506-555-0101','email':'alice@email.com'},'insurance':{'plan':'premium','id':'INS-00123'}}),
   json.dumps({'conditions':['Hypertension','Hyperlipidaemia'],'allergies':[],'vitals':[{'date':'2023-04-10','bp':'142/88','weight_kg':72},{'date':'2023-10-01','bp':'128/82','weight_kg':70}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P002','Bob Chen',
   json.dumps({'age':52,'dob':'1972-07-22','province':'ON','contact':{'phone':'416-555-2233','email':'bob@email.com'},'insurance':{'plan':'standard','id':'INS-00456'}}),
   json.dumps({'conditions':['Hypertension','Type 2 Diabetes'],'allergies':['Penicillin'],'vitals':[{'date':'2023-05-15','bp':'148/92','weight_kg':88},{'date':'2024-01-10','bp':'138/86','weight_kg':86}],'smoker':True}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':True},'portal_enabled':True})),
  ('P003','Fatima Rashid',
   json.dumps({'age':38,'dob':'1986-11-05','province':'BC','contact':{'phone':'604-555-9900','email':'fatima@email.com'},'insurance':{'plan':'premium','id':'INS-00789'}}),
   json.dumps({'conditions':['Asthma','Obesity'],'allergies':['Sulfa drugs','NSAIDs'],'vitals':[{'date':'2023-08-20','bp':'148/92','weight_kg':95}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':False,'sms':True},'portal_enabled':False})),
  ('P004','James MacLeod',
   json.dumps({'age':61,'dob':'1963-01-30','province':'NS','contact':{'phone':'902-555-7788','email':'james@email.com'},'insurance':{'plan':'standard','id':'INS-01011'}}),
   json.dumps({'conditions':['Type 2 Diabetes'],'allergies':[],'vitals':[{'date':'2023-09-01','bp':'118/76','weight_kg':80}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P005','Mei-Ling Tran',
   json.dumps({'age':58,'dob':'1966-08-19','province':'QC','contact':{'phone':'514-555-1122','email':'mei@email.com'},'insurance':{'plan':'premium','id':'INS-01213'}}),
   json.dumps({'conditions':['Type 2 Diabetes','Hypertension','CKD Stage 3'],'allergies':['Metformin'],'vitals':[{'date':'2023-11-10','bp':'155/96','weight_kg':65},{'date':'2024-02-01','bp':'145/90','weight_kg':64}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':True,'sms':True},'portal_enabled':True})),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", patients)

lab_rows = [
  ('P001','2023-10-01', json.dumps([
    {'test':'HbA1c','value':7.2,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':68,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'},
    {'test':'LDL','value':3.1,'unit':'mmol/L','ref_range':'<2.6','flag':'H'}])),
  ('P002','2024-01-10', json.dumps([
    {'test':'HbA1c','value':8.4,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':55,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Creatinine','value':115,'unit':'umol/L','ref_range':'62-106','flag':'H'}])),
  ('P004','2023-09-01', json.dumps([
    {'test':'HbA1c','value':7.8,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':72,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'}])),
  ('P005','2024-02-01', json.dumps([
    {'test':'HbA1c','value':9.1,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':38,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Potassium','value':5.8,'unit':'mmol/L','ref_range':'3.5-5.0','flag':'H'}])),
]
conn.executemany("INSERT INTO lab_results (patient_id,result_date,tests) VALUES (?,?,?)", lab_rows)

med_rows = [
  ('P001','2023-01-15', json.dumps({'drug':'Lisinopril','dose':'10mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['dizziness','dry cough']})),
  ('P001','2023-01-15', json.dumps({'drug':'Atorvastatin','dose':'40mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['myalgia']})),
  ('P002','2022-06-01', json.dumps({'drug':'Metformin','dose':'500mg','frequency':'BID','prescriber':'Dr. Pham','side_effects':['nausea','diarrhoea']})),
  ('P002','2022-06-01', json.dumps({'drug':'Lisinopril','dose':'5mg','frequency':'once daily','prescriber':'Dr. Pham','side_effects':[]})),
  ('P003','2021-09-10', json.dumps({'drug':'Salbutamol','dose':'2.5mg','frequency':'PRN','prescriber':'Dr. Singh','side_effects':['tremor','palpitations']})),
  ('P005','2023-03-01', json.dumps({'drug':'Insulin glargine','dose':'20 units','frequency':'nocte','prescriber':'Dr. Pham','side_effects':['hypoglycaemia']})),
]
conn.executemany("INSERT INTO medications (patient_id,prescribed,details) VALUES (?,?,?)", med_rows)
conn.commit()

print("Healthcare JSON dataset ready:")
for tbl in ("patients","lab_results","medications"):
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl}: {n} rows")

# Preview one JSON column
row = conn.execute("SELECT full_name, demographics FROM patients LIMIT 1").fetchone()
print(f"\nSample demographics JSON for {row['full_name']}:")
print(json.dumps(json.loads(row['demographics']), indent=2))


import json

print("=== PostgreSQL JSONB operators ===")
print()

# Simulate with SQLite + Python since no PostgreSQL available
# Show all operators with annotated examples

operators = [
    ("->",   "Get JSON object field by key (returns JSONB)",
     "col->'contact'",             "Returns the contact object as JSONB"),
    ("->>",  "Get JSON object field as TEXT",
     "col->>'province'",           "Returns 'NB' as text (castable to other types)"),
    ("->",   "Get JSON array element by index",
     "col->'vitals'->0",           "Returns first vitals entry as JSONB"),
    ("->>",  "Get JSON array element as TEXT",
     "col->'vitals'->>0",          "Returns first vitals entry as text (JSON string)"),
    ("#>",   "Get value at path (returns JSONB)",
     "col#>'{contact,phone}'",     "Same as col->'contact'->'phone'"),
    ("#>>",  "Get value at path as TEXT",
     "col#>>'{contact,phone}'",    "Same as col->'contact'->>'phone'"),
    ("@>",   "Does left contain right? (returns boolean)",
     "col @> '{\"smoker\":false}'", "True if smoker=false is in the JSON"),
    ("<@",   "Is left contained in right?",
     "'{\"a\":1}' <@ col",         "True if {a:1} is a subset of col"),
    ("?",    "Does key exist at top level?",
     "col ? 'allergies'",          "True if key 'allergies' exists"),
    ("?|",   "Any of these keys exist?",
     "col ?| ARRAY['email','sms']","True if email OR sms key present"),
    ("?&",   "All of these keys exist?",
     "col ?& ARRAY['email','sms']","True if email AND sms both present"),
    ("||",   "Concatenate two JSONB values",
     "col || '{\"new_key\":1}'",    "Merges; right-side wins on duplicate keys"),
    ("-",    "Delete key from JSONB object",
     "col - 'temp_field'",         "Returns JSONB with key removed"),
    ("#-",   "Delete at path",
     "col #- '{contact,fax}'",     "Removes nested key"),
]

print(f"  {'Operator':<6s}  {'Purpose':<38s}  {'Example':<36s}  Notes")
print("  " + "-"*100)
for row in operators:
    op, purpose, example, note = row
    print(f"  {op:<6s}  {purpose:<38s}  {example:<36s}  {note}")


---
## @> containment: the most powerful JSONB operator

In [ ]:
import sqlite3, json

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

# Healthcare dataset: patients with JSON clinical data
conn.executescript("""
CREATE TABLE patients (
    patient_id   TEXT PRIMARY KEY,
    full_name    TEXT NOT NULL,
    demographics JSON NOT NULL,   -- age, dob, province, contact info
    clinical     JSON NOT NULL,   -- conditions, allergies, vitals history
    preferences  JSON             -- notification prefs, language, etc.
);

CREATE TABLE lab_results (
    lab_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    result_date TEXT NOT NULL,
    tests       JSON NOT NULL    -- array of {test, value, unit, ref_range, flag}
);

CREATE TABLE medications (
    med_id      INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id  TEXT NOT NULL,
    prescribed  TEXT NOT NULL,
    details     JSON NOT NULL    -- drug, dose, frequency, prescriber, side_effects
);
""")

patients = [
  ('P001','Alice Ngata',
   json.dumps({'age':45,'dob':'1980-03-15','province':'NB','contact':{'phone':'506-555-0101','email':'alice@email.com'},'insurance':{'plan':'premium','id':'INS-00123'}}),
   json.dumps({'conditions':['Hypertension','Hyperlipidaemia'],'allergies':[],'vitals':[{'date':'2023-04-10','bp':'142/88','weight_kg':72},{'date':'2023-10-01','bp':'128/82','weight_kg':70}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P002','Bob Chen',
   json.dumps({'age':52,'dob':'1972-07-22','province':'ON','contact':{'phone':'416-555-2233','email':'bob@email.com'},'insurance':{'plan':'standard','id':'INS-00456'}}),
   json.dumps({'conditions':['Hypertension','Type 2 Diabetes'],'allergies':['Penicillin'],'vitals':[{'date':'2023-05-15','bp':'148/92','weight_kg':88},{'date':'2024-01-10','bp':'138/86','weight_kg':86}],'smoker':True}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':True},'portal_enabled':True})),
  ('P003','Fatima Rashid',
   json.dumps({'age':38,'dob':'1986-11-05','province':'BC','contact':{'phone':'604-555-9900','email':'fatima@email.com'},'insurance':{'plan':'premium','id':'INS-00789'}}),
   json.dumps({'conditions':['Asthma','Obesity'],'allergies':['Sulfa drugs','NSAIDs'],'vitals':[{'date':'2023-08-20','bp':'148/92','weight_kg':95}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':False,'sms':True},'portal_enabled':False})),
  ('P004','James MacLeod',
   json.dumps({'age':61,'dob':'1963-01-30','province':'NS','contact':{'phone':'902-555-7788','email':'james@email.com'},'insurance':{'plan':'standard','id':'INS-01011'}}),
   json.dumps({'conditions':['Type 2 Diabetes'],'allergies':[],'vitals':[{'date':'2023-09-01','bp':'118/76','weight_kg':80}],'smoker':False}),
   json.dumps({'language':'en','notifications':{'email':True,'sms':False},'portal_enabled':True})),
  ('P005','Mei-Ling Tran',
   json.dumps({'age':58,'dob':'1966-08-19','province':'QC','contact':{'phone':'514-555-1122','email':'mei@email.com'},'insurance':{'plan':'premium','id':'INS-01213'}}),
   json.dumps({'conditions':['Type 2 Diabetes','Hypertension','CKD Stage 3'],'allergies':['Metformin'],'vitals':[{'date':'2023-11-10','bp':'155/96','weight_kg':65},{'date':'2024-02-01','bp':'145/90','weight_kg':64}],'smoker':False}),
   json.dumps({'language':'fr','notifications':{'email':True,'sms':True},'portal_enabled':True})),
]
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", patients)

lab_rows = [
  ('P001','2023-10-01', json.dumps([
    {'test':'HbA1c','value':7.2,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':68,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'},
    {'test':'LDL','value':3.1,'unit':'mmol/L','ref_range':'<2.6','flag':'H'}])),
  ('P002','2024-01-10', json.dumps([
    {'test':'HbA1c','value':8.4,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':55,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Creatinine','value':115,'unit':'umol/L','ref_range':'62-106','flag':'H'}])),
  ('P004','2023-09-01', json.dumps([
    {'test':'HbA1c','value':7.8,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':72,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'N'}])),
  ('P005','2024-02-01', json.dumps([
    {'test':'HbA1c','value':9.1,'unit':'%','ref_range':'<5.7','flag':'H'},
    {'test':'eGFR','value':38,'unit':'mL/min/1.73m2','ref_range':'>60','flag':'L'},
    {'test':'Potassium','value':5.8,'unit':'mmol/L','ref_range':'3.5-5.0','flag':'H'}])),
]
conn.executemany("INSERT INTO lab_results (patient_id,result_date,tests) VALUES (?,?,?)", lab_rows)

med_rows = [
  ('P001','2023-01-15', json.dumps({'drug':'Lisinopril','dose':'10mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['dizziness','dry cough']})),
  ('P001','2023-01-15', json.dumps({'drug':'Atorvastatin','dose':'40mg','frequency':'once daily','prescriber':'Dr. Lee','side_effects':['myalgia']})),
  ('P002','2022-06-01', json.dumps({'drug':'Metformin','dose':'500mg','frequency':'BID','prescriber':'Dr. Pham','side_effects':['nausea','diarrhoea']})),
  ('P002','2022-06-01', json.dumps({'drug':'Lisinopril','dose':'5mg','frequency':'once daily','prescriber':'Dr. Pham','side_effects':[]})),
  ('P003','2021-09-10', json.dumps({'drug':'Salbutamol','dose':'2.5mg','frequency':'PRN','prescriber':'Dr. Singh','side_effects':['tremor','palpitations']})),
  ('P005','2023-03-01', json.dumps({'drug':'Insulin glargine','dose':'20 units','frequency':'nocte','prescriber':'Dr. Pham','side_effects':['hypoglycaemia']})),
]
conn.executemany("INSERT INTO medications (patient_id,prescribed,details) VALUES (?,?,?)", med_rows)
conn.commit()

print("Healthcare JSON dataset ready:")
for tbl in ("patients","lab_results","medications"):
    n = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl}: {n} rows")

# Preview one JSON column
row = conn.execute("SELECT full_name, demographics FROM patients LIMIT 1").fetchone()
print(f"\nSample demographics JSON for {row['full_name']}:")
print(json.dumps(json.loads(row['demographics']), indent=2))


print("=== @> containment: the most powerful JSONB operator ===")
print()

# Simulate @> with Python on our SQLite JSON data
import json

def jsonb_contains(doc_str, filter_str):
    """Simulate PostgreSQL JSONB @> containment."""
    doc    = json.loads(doc_str) if isinstance(doc_str, str) else doc_str
    filter_ = json.loads(filter_str) if isinstance(filter_str, str) else filter_str
    if isinstance(filter_, dict):
        return all(
            k in doc and jsonb_contains(doc[k], v)
            for k, v in filter_.items()
        )
    elif isinstance(filter_, list):
        return all(any(jsonb_contains(d, f) for d in doc) for f in filter_)
    else:
        return doc == filter_

# Test on our patients
rows = conn.execute("SELECT patient_id, full_name, demographics, clinical FROM patients").fetchall()

print("Simulating: clinical @> '{\"smoker\": true}'")
for r in rows:
    if jsonb_contains(r['clinical'], '{"smoker": true}'):
        print(f"  MATCH  {r['full_name']}")

print()
print("Simulating: demographics @> '{\"province\": \"NB\"}'")
for r in rows:
    if jsonb_contains(r['demographics'], '{"province": "NB"}'):
        print(f"  MATCH  {r['full_name']}")

print()
print("Simulating: demographics @> '{\"insurance\": {\"plan\": \"premium\"}}'")
for r in rows:
    if jsonb_contains(r['demographics'], '{"insurance": {"plan": "premium"}}'):
        print(f"  MATCH  {r['full_name']}")

print()
print("PostgreSQL @> queries (GIN-indexable — efficient on large datasets):")
pg_contains = [
    ("WHERE clinical @> '{\"smoker\":true}'",
     "Find all smokers"),
    ("WHERE demographics @> '{\"province\":\"NB\"}'",
     "Find patients in NB"),
    ("WHERE demographics @> '{\"insurance\":{\"plan\":\"premium\"}}'",
     "Nested containment: premium plan"),
    ("WHERE clinical @> '{\"conditions\":[\"Hypertension\"]}'",
     "Array containment: has Hypertension in conditions list"),
    ("WHERE clinical @> '{\"allergies\":[\"Penicillin\"]}'",
     "Has Penicillin allergy"),
]
for query, desc in pg_contains:
    print(f"  {desc}:")
    print(f"    {query}")
    print()


---
## JSONB indexing strategies

In [ ]:
print("=== JSONB indexing strategies ===")
print()

index_types = [
    ("GIN (default)", "jsonb_ops",
     "CREATE INDEX idx_clinical_gin ON patients USING GIN (clinical);",
     "@>, ?, ?|, ?& operators\nSlower to build; larger; catches all JSON operators"),
    ("GIN (path_ops)", "jsonb_path_ops",
     "CREATE INDEX idx_clinical_path ON patients USING GIN (clinical jsonb_path_ops);",
     "@> operator only\nFaster to build; smaller; most common choice"),
    ("B-tree on extracted value", "expression index",
     "CREATE INDEX idx_province ON patients ((demographics->>'province'));",
     "= and range on specific extracted text values\nFastest for equality on known key"),
    ("B-tree on cast value", "expression index with cast",
     "CREATE INDEX idx_age ON patients (((demographics->>'age')::int));",
     "Numeric comparisons on extracted values"),
    ("Partial GIN", "filtered GIN",
     "CREATE INDEX idx_high_risk ON patients USING GIN (clinical)\n  WHERE (clinical->>'smoker')::bool = true;",
     "GIN on subset of rows; smaller index for selective queries"),
]

for name, ops_type, ddl, use_case in index_types:
    print(f"  {name} ({ops_type}):")
    print(f"    DDL:      {ddl.splitlines()[0]}")
    if '\n' in ddl:
        print(f"              {ddl.splitlines()[1]}")
    print(f"    Use when: {use_case.splitlines()[0]}")
    print()

print("Index selection guide:")
decisions = [
    ("Filter by specific key=value",    "B-tree expression index on col->>'key'"),
    ("Containment @> (general)",        "GIN jsonb_path_ops (smaller, fast for @>)"),
    ("Key existence ? or ?|",           "GIN jsonb_ops (jsonb_path_ops does not support ?)"),
    ("Numeric range on JSON field",     "B-tree on (col->>'key')::numeric"),
    ("Text search inside JSON",         "GIN trgm index on col->>'text_field'"),
    ("All JSON operators needed",       "GIN jsonb_ops (most permissive, largest index)"),
]
for need, solution in decisions:
    print(f"  {need:<38s}  {solution}")


---
## Common Pitfalls

**1. Using JSON instead of JSONB for queried data**
PostgreSQL's `JSON` type stores the raw text verbatim; it cannot be indexed with GIN and operators like `@>` and `?` are not available. Use `JSONB` for any column you filter or index on. `JSON` is appropriate only for storing data that is consumed verbatim (e.g., audit logs where exact key order matters).

**2. `->` vs `->>` returning wrong type**
`col->'key'` returns JSONB (including quotes for strings: `"NB"`). `col->>'key'` returns TEXT (`NB`). `WHERE col->'province' = 'NB'` always returns false because a JSONB string `"NB"` is not equal to the SQL literal `'NB'`. Always use `->>` for scalar comparisons.

**3. @> containment with wrong JSONB literal syntax**
`col @> '{province: "NB"}'` is invalid (keys must be quoted in JSON). Use `col @> '{"province": "NB"}'::jsonb`. The `::jsonb` cast also validates the literal at query time rather than silently returning no results.

**4. GIN index not being used for `->>`**
A GIN index on `clinical` does not help `WHERE clinical->>'smoker' = 'true'`. GIN indexes help `@>`, `?`, `?|`, `?&`. For `->>` equality, create an expression B-tree index: `CREATE INDEX ON patients ((clinical->>'smoker'))`.

**5. jsonb_path_ops GIN not supporting `?` operator**
`jsonb_path_ops` GIN index only supports `@>`. If you also need `?` (key existence), use the default `jsonb_ops`. If you only need `@>`, use `jsonb_path_ops` for a smaller, faster index.

**6. Updating JSONB without returning the modified value**
`UPDATE patients SET clinical = jsonb_set(clinical, '{smoker}', 'false') WHERE patient_id = 'P001'` — forgetting `RETURNING clinical` means you cannot verify the update in the same statement. Always add `RETURNING` for auditable updates.


---
*sql_methods_library - Samantha McGarrigle*